In [2]:
import fs from 'node:fs';
import path from 'node:path';
import { ChatOpenAI } from '@langchain/openai';
import { OPENAI_API_KEY } from './src/lib/vars.mjs';
import * as z from 'zod';

function logFile(logContent, fileName = 'jupyter.md') {
  const logFileName = `logs/${fileName}`;
  const logDir = path.dirname(logFileName);
  if (!fs.existsSync(logDir)) {
    fs.mkdirSync(logDir, { recursive: true });
  }
  fs.writeFileSync(logFileName, '```markdown\n' + logContent + '\n```', 'utf8');
  return logContent;
}

const gpt5 = new ChatOpenAI({
  modelName: 'gpt-5',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt4oMini = new ChatOpenAI({
  modelName: 'gpt-4o-mini',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});
const gpt5Nano = new ChatOpenAI({
  modelName: 'gpt-5-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt41Nano = new ChatOpenAI({
  modelName: 'gpt-4.1-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});

async function queryAllModels(prompt, callFunction) {
  console.log(`Started: ${new Date()}\n`);
  return Promise.all([
    callFunction(prompt, gpt4oMini),
    callFunction(prompt, gpt41Nano),
    callFunction(prompt, gpt5Nano),
    // callFunction(prompt, gpt5),
  ]);
}

## Rewrite Initial Query


### Query Rewrite

**Results:** 

All models performed satisfactory with similar times.

In [ ]:
import { rewriteQuery } from './src/lib/rag.mjs';

const redditQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`

async function sanitizeQuery(question, model) {
  const { answer: query } = await rewriteQuery(question, model);
  console.log(`**${model.model}** – ${new Date()}:\n${query}\n`);
  return query;
}

queryAllModels(redditQuestion, sanitizeQuery)

Started: Thu Aug 21 2025 16:10:29 GMT-0400 (Eastern Daylight Time)



Promise { <pending> }

gpt-4o-mini – Thu Aug 21 2025 16:10:40 GMT-0400 (Eastern Daylight Time):
As an exchange student planning to study in Canada for one academic year, my study permit application was submitted around mid-June with processing times reportedly changing from six weeks to ten weeks. We plan to depart by August 29 for a September 2 start date, and I cannot arrive late. If the study permit is not issued before August 29, what options do I have, including traveling to Canada on a visitor visa and obtaining the study permit after arrival? What are the potential consequences for re-entry to Canada after Christmas if I entered on a visitor visa? What steps should I take and what questions should I ask IRCC to proceed safely?

gpt-5 – Thu Aug 21 2025 16:10:40 GMT-0400 (Eastern Daylight Time):
What options do I have as an exchange student planning to study in Canada for one academic year when my study permit application is likely to be processed after my planned departure date (August 29) for a Septem

### Query Extraction

#### Decomposing Main Query

#### Test Results

| Model       | Response Time | Performance | Notes                                                   |
| ----------- | ------------- | ----------- | ------------------------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        |                                                         |
| GPT-4.1     | < 5 sec       | Good        |                                                         |
| GPT-5-nano  | < 30 sec      | Mediocre    | Questions need further break down.                      |
| GPT-5       | > 1 min       | Ok          | Questions well-thought but must be broken down further. |


In [ ]:
const compoundQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`;

const questionExtractionPrompt = `You are an assistant that prepares user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration rules.
The user may provide a long, informal story or question. Your task is:
1. Identify all explicit and implicit questions they are asking.  
2. Rewrite each one as a clear, self-contained question that could be answered directly from IRCC documentation.  
3. Condense the result into the *smallest possible set of non-overlapping, atomic questions* that fully capture the user’s intent.  
4. Eliminate redundancy — avoid rephrasing the same issue multiple times.  
5. Do not provide answers — only the minimal list of questions.

**User question:**

\`\`\`
${compoundQuestion}
\`\`\`
`;

async function extractQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z.array(z.string()).describe('A list of questions derived from the user query'),
      })
    )
    .invoke(q);
  const content = response.questions.map(q => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

const extractedQuestions = await queryAllModels(questionExtractionPrompt, extractQuestions);


Started: Thu Aug 21 2025 15:47:28 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** Thu Aug 21 2025 15:47:29 GMT-0400 (Eastern Daylight Time):
	- Can I enter Canada as a student with a valid study permit obtained before my planned arrival date?
	- Is it possible to enter Canada as a tourist and then switch to a student status after arrival?
	- Will entering Canada on a tourist visa affect my ability to study or change my immigration status later?
	- What are the risks of traveling to Canada on a tourist visa if my study permit has not yet been approved?
	- Can I leave Canada during my studies and re-enter on a tourist visa without affecting my student status?

**gpt-4o-mini** Thu Aug 21 2025 15:47:32 GMT-0400 (Eastern Daylight Time):
	- What are the current processing times for a study permit application for Canada?
	- Can I enter Canada on a tourist visa while waiting for my study permit?
	- Will entering Canada on a tourist visa affect my ability to return after traveling back home 

#### Identifying Key Questions

##### Test Summary

| Model       | Response Time | Performance | Notes                                  |
| ----------- | ------------- | ----------- | -------------------------------------- |
| 🥇 GPT-4.1     | < 5 sec       | Good        | Correctly identified the key question. |
| GPT-4o-mini | < 5 sec       | Good        | Included some secondary questions.     |
| GPT-5-nano  | < 30 sec      | Mediocre    | Included the most questions.           |
| GPT-5       | < 30 sec       | Ok          | Included some secondary questions.     |


In [ ]:

const mdExtractedQuestions = extractedQuestions[1].questions.map(q => `\t- ${q}`).join('\n');
const questionDiscriminationPrompt = `You are helping prepare user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration.
Input: a list of atomic questions generated from a user’s long query.
Task:
1. Identify the key question(s) that directly capture the user’s main intent.  
   - Keep only the questions that must be answered to resolve the user’s core concern.  
   - Discard questions that are secondary, conditional, or only relevant as follow-ups.
2. Output only the minimal set of key questions, without explanation, ranked by relevance to the user query.
Important: The result should be as short as possible while still fully representing the original user’s primary intent.

User query:

\`\`\`
${compoundQuestion}
\`\`\`

List of questions:
\`\`\`
${mdExtractedQuestions}
\`\`\`
`

async function discriminateQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z.array(z.string()).describe('A list of key questions derived from the user query'),
      })
    )
    .invoke(q);
  const content = response.questions.map(q => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

console.log(`\n\nDiscriminating questions for: "${compoundQuestion}"\n`);
console.log(`Extracted questions:\n${mdExtractedQuestions}\n`);
queryAllModels(questionDiscriminationPrompt, discriminateQuestions);



Discriminating questions for: "Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if th

## Vector Search

### Retrieval

In [3]:
import { vectorSearch, chunksToMarkdown } from './src/lib/vector-search.mjs';
const retrieveQuery = 'Can I enter Canada on a tourist visa while waiting for my study permit?'
const chunks = await vectorSearch(retrieveQuery);
logFile(chunksToMarkdown(chunks), 'chunks.md');
chunks

[
  {
    text: "# Extend your study permit or restore your status\n" +
      "\n" +
      "*   [When to apply](/en/immigration-refugees-citizenship/services/study-canada/extend-study-permit/when-to-apply.html) [When to apply](/en/immigration-refugees-citizenship/services/study-canada/extend-study-permit/when-to-apply.html#gc-document-nav)\n" +
      "*   [How to apply](#)\n" +
      "*   [What to do if your permit expired](/en/immigration-refugees-citizenship/services/study-canada/extend-study-permit/expired-permit.html) [What to do if your permit expired](/en/immigration-refugees-citizenship/services/study-canada/extend-study-permit/expired-permit.html#gc-document-nav)\n" +
      "\n" +
      "# How to apply\n" +
      "\n" +
      "In Canada study permit applicants: New rules about applying at a port of entry\n" +
      "\n" +
      "Most foreign nationals **already in Canada** can [no longer apply for a study permit at a port of entry](/en/border-services-agency/news/2024/12/ending

### Reference Discrimination

#### Test Summary

| Model          | Response Time | Performance | Notes                                 |
| -------------- | ------------- | ----------- | ------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        | Picked the most chunks, all relevant. |
| GPT-4.1        | < 5 sec       | Ok          |                                       |
| GPT-5          | < 1 min       | Ok          |                                       |
| GPT-5-nano     | < 30 sec      | Mediocre    | Missed a highly relevant chunk.       |


In [4]:
const chunkDiscriminationPrompt = `I'll give you a markdown content with a list of results from a vector search for a question.
Select only the references that help to answer th question.
Pay attention to the header of the top-most header in each reference to identify if it's related to the topic we want to answer; discard it if it is not.
Return an array containing the selected references' numbers.

**Question:** ${retrieveQuery}

**Chunks:**

\`\`\`markdown
${chunksToMarkdown(chunks)}
\`\`\`
`;

async function discriminateReferences(prompt, model) {
  const { references: referenceIndexes } = await model
  .withStructuredOutput(
    z.object({
      references: z.array(z.number()).describe('An array of numbers representing the selected references from the chunks'),
    })
  )
  .invoke(prompt);

  console.log(`**${model.model}** – ${new Date()}: ${JSON.stringify(referenceIndexes)}`);
  return referenceIndexes;
}

const referenceIndexes = await queryAllModels(chunkDiscriminationPrompt, discriminateReferences);

Started: Thu Aug 21 2025 19:30:07 GMT-0400 (Eastern Daylight Time)

**gpt-4o-mini** – Thu Aug 21 2025 19:30:08 GMT-0400 (Eastern Daylight Time): [2,3,4,6]
**gpt-4.1-nano** – Thu Aug 21 2025 19:30:08 GMT-0400 (Eastern Daylight Time): [2,6]
**gpt-5-nano** – Thu Aug 21 2025 19:30:18 GMT-0400 (Eastern Daylight Time): [3,6]


## Generating Answer

### Test Summary

| Model          | Response Time | Performance | Notes                             |
| -------------- | ------------- | ----------- | --------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        | Concise and accurate.             |
| GPT-5-nano     | < 30 sec      | Good          | Good structure. Too many details. |
| GPT-4.1        | < 5 sec       | Ok          |                                   |
| GPT-5          | < 1 min       | Ok          | Good structure. Too many details. |


In [5]:
import { generateAnswer } from './src/lib/rag.mjs';

async function generateRAGAnswer(query, model) {
  const selectChunks = chunks.filter((_, i) => referenceIndexes[1].includes(i + 1));
  const references = chunksToMarkdown(selectChunks);
  const { answer } = await generateAnswer(query, references, model);
  console.log(`**${model.model}** – ${new Date()}:\n${answer}\n\n* * *\n`);
  return answer;
}

console.log(`"${retrieveQuery}"\n`);
await queryAllModels(retrieveQuery, generateRAGAnswer)

"Can I enter Canada on a tourist visa while waiting for my study permit?"

Started: Thu Aug 21 2025 19:30:18 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Thu Aug 21 2025 19:30:20 GMT-0400 (Eastern Daylight Time):
Yes, it is possible to enter Canada on a visitor visa (temporary resident visa) while waiting for your study permit, provided you meet the entry requirements and have the appropriate documentation. When arriving at the port of entry, you should inform the border services officer that you intend to study in Canada and present your letter of introduction or visa, as applicable. If you are from a country that requires an eTA, ensure it is valid and linked to your passport. Keep in mind that your ability to enter Canada depends on the officer's assessment of your eligibility at the border, and having a visitor visa does not guarantee entry for the purpose of studying. Additionally, if you are already in Canada and have applied for your study permit from within the country,

[
  "Yes, you can enter Canada on a tourist visa while waiting for your study permit, provided you meet the entry requirements. If your study permit application is still being processed and you are not yet in Canada, you may still be eligible to enter as a visitor. However, you must ensure that you have the appropriate visa or electronic travel authorization (eTA) linked to your passport, depending on your nationality. If you are from a country that requires a visitor visa, it will be in your passport and will indicate whether you can enter Canada once or multiple times. It is crucial to enter Canada before your visa expires [[1](https://www.canada.ca/en/immigration-refugees-citizenship/services/study-canada/study-permit/after-apply-next-steps.html)][[2](https://www.canada.ca/en/immigration-refugees-citizenship/services/study-canada/study-permit/apply.html)].",
  "Yes, it is possible to enter Canada on a visitor visa (temporary resident visa) while waiting for your study permit, provid